# Установка библиотек

In [195]:
# %pip install pandas numpy plotly nbformat scikit-learn

# Импорты

In [196]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import abc as abc
import sklearn as skl

# Дата сет

## Загружаем

In [197]:
df = pd.read_csv('backpack.csv')


df_clean = df.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [198]:
df.describe()

,id,Compartments,Weight Capacity (kg),Price
count,300000.000000,300000.000000,299862.000000,300000.000000
mean,149999.500000,5.443590,18.029994,81.411107
std,86602.684716,2.890766,6.966914,39.039340
min,0.000000,1.000000,5.000000,15.000000
25%,74999.750000,3.000000,12.097867,47.384620
50%,149999.500000,5.000000,18.068614,80.956120
75%,224999.250000,8.000000,24.002375,115.018160
max,299999.000000,10.000000,30.000000,150.000000


In [199]:
df.head()

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312


## Пред обработка

In [200]:
DROP_COLUMNS = ['id']

df.drop(DROP_COLUMNS, axis=1, inplace=True)

In [201]:
CATEGORICAL_COLUMNS = ['Brand', 'Material', 'Size', 'Laptop Compartment', 'Waterproof', 'Style', 'Color']

for column in CATEGORICAL_COLUMNS:
    mode_value = df[column].mode()[0]
    df[column] = df[column].fillna(mode_value)
    


In [202]:
MEDIANA_COLUMNS = ['Weight Capacity (kg)']

for column in MEDIANA_COLUMNS:
    median_value = df[column].median()
    df[column] = df[column].fillna(median_value)

In [203]:
ONE_HOT_COLUMNS = ['Brand', 'Material', 'Size', 'Style', 'Color']
LABEL_ENCODE_COLUMNS = ['Waterproof' , 'Laptop Compartment']

df = pd.get_dummies(df, columns=ONE_HOT_COLUMNS, drop_first=True)

for col in LABEL_ENCODE_COLUMNS:
    df[f'{col}_Encoded'] = skl.preprocessing.LabelEncoder().fit_transform(df[col])
    df.drop(col, axis=1, inplace=True)


In [204]:
df.head()

,Compartments,Weight Capacity (kg),Price,Brand_Jansport,Brand_Nike,Brand_Puma,Brand_Under Armour,Material_Leather,Material_Nylon,Material_Polyester,...,Size_Small,Style_Messenger,Style_Tote,Color_Blue,Color_Gray,Color_Green,Color_Pink,Color_Red,Waterproof_Encoded,Laptop Compartment_Encoded
0,7.0,11.611723,112.15875,True,False,False,False,True,False,False,...,False,False,True,False,False,False,False,False,0,1
1,10.0,27.078537,68.88056,True,False,False,False,False,False,False,...,True,True,False,False,False,True,False,False,1,1
2,2.0,16.643760,39.17320,False,False,False,True,True,False,False,...,True,True,False,False,False,False,False,True,0,1
3,8.0,12.937220,80.60793,False,True,False,False,False,True,False,...,True,True,False,False,False,True,False,False,0,1
4,1.0,17.749338,86.02312,False,False,False,False,False,False,False,...,False,True,False,False,False,True,False,False,1,1


In [205]:
df.isna().sum()

Compartments                  0
Weight Capacity (kg)          0
Price                         0
Brand_Jansport                0
Brand_Nike                    0
Brand_Puma                    0
Brand_Under Armour            0
Material_Leather              0
Material_Nylon                0
Material_Polyester            0
Size_Medium                   0
Size_Small                    0
Style_Messenger               0
Style_Tote                    0
Color_Blue                    0
Color_Gray                    0
Color_Green                   0
Color_Pink                    0
Color_Red                     0
Waterproof_Encoded            0
Laptop Compartment_Encoded    0
dtype: int64

In [206]:
# X_train , X_test , y_train  , y_test  = skl.model_selection.train_test_split(
#     df.drop('Price', axis=1),
#     df['Price'],
#     test_size=0.2,
#     random_state=42
# )

In [207]:
fig_hist = px.histogram(
    df, 
    x='Price', 
    title='Распределение цены',
    nbins=50, 
    marginal="box" 
)

# Добавление подписей осей
fig_hist.update_layout(
    xaxis_title="Цена",
    yaxis_title="Количество товаров"
)

fig_hist.show()

In [208]:


def plot_mean_by_category(df, column_name, top_n=10):
    """
    Строит столбчатую диаграмму средней цены по заданной категориальной колонке.

    Параметры:
    - df (pd.DataFrame): Исходный DataFrame.
    - column_name (str): Название категориальной колонки для анализа.
    - top_n (int): Отображать только N самых дорогих категорий (для читаемости).
    """
    
    # 1. Группировка и сортировка
    category_mean = df.groupby(column_name)["Price"].mean().reset_index()
    category_mean = category_mean.sort_values("Price", ascending=False)
    
    # Обрезка: берем только TOP N для наглядности, если категорий много
    if len(category_mean) > top_n:
        category_mean = category_mean.head(top_n)
        dynamic_title = f"Средняя цена: TOP {top_n} по {column_name}"
    else:
        dynamic_title = f"Средняя цена по категории '{column_name}'"


    fig = px.bar(
        category_mean,
        x=column_name, 
        y="Price",
        title=f"{dynamic_title}",
        text="Price",
        color="Price",
        color_continuous_scale="Viridis"
    )
    
    fig.update_layout(
        yaxis_title="Средняя цена",
        xaxis_title=column_name,
    )
    

    fig.update_traces(
        texttemplate='$%{text:.2f}', 
        textposition='outside'
    )
    
    fig.show()

# --- Использование функции ---

mean_columns = ['Brand', 'Material', 'Size', 'Style']

for column in mean_columns:
    # Устанавливаем лимит для 'Brand' (например, 15), чтобы избежать перегрузки графика
    if column == 'Brand':
        plot_mean_by_category(df_clean, column, top_n=15)
    else:
        plot_mean_by_category(df_clean, column, top_n=10)

# Линейная регрессия

In [209]:
FEATURES_COLUMNS_FOR_LINGREG = df.columns.tolist()
FEATURES_COLUMNS_FOR_LINGREG.remove('Price')


assert all(col in df.columns for col in FEATURES_COLUMNS_FOR_LINGREG), "Some columns are missing in the DataFrame."
assert "Price" not in FEATURES_COLUMNS_FOR_LINGREG, "Target column 'Price' is features."

X_train , X_test , y_train  , y_test  = skl.model_selection.train_test_split(
    df[FEATURES_COLUMNS_FOR_LINGREG],
    df['Price'],
    test_size=0.2,
    random_state=42
)


lin_reg = skl.linear_model.LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

mse = skl.metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2_score = skl.metrics.r2_score(y_test, y_pred)


metrics_data = {
    'Метрика': ['RMSE', 'MSE', 'R^2 Score'],
    'Значение': [rmse, mse, r2_score]
}
pd.DataFrame(metrics_data)


,Метрика,Значение
0,RMSE,38.920756
1,MSE,1514.825235
2,R^2 Score,0.001181


# Randon Forest

In [210]:

FEATURES_COLUMNS_FOR_RANDOM_FOREST = df.columns.tolist()
FEATURES_COLUMNS_FOR_RANDOM_FOREST.remove('Price')


assert all(col in df.columns for col in FEATURES_COLUMNS_FOR_RANDOM_FOREST), "Some columns are missing in the DataFrame."
assert "Price" not in FEATURES_COLUMNS_FOR_RANDOM_FOREST, "Target column 'Price' is features."

X_train , X_test , y_train  , y_test  = skl.model_selection.train_test_split(
    df[FEATURES_COLUMNS_FOR_RANDOM_FOREST],
    df['Price'],
    test_size=0.2,
    random_state=42
)


rf_reg = skl.ensemble.RandomForestRegressor(n_estimators=128, random_state=42,)
rf_reg.fit(X_train, y_train) 

y_pred = rf_reg.predict(X_test)

mse = skl.metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2_score = skl.metrics.r2_score(y_test, y_pred)


metrics_data = {
    'Метрика': ['RMSE', 'MSE', 'R^2 Score'],
    'Значение': [rmse, mse, r2_score]
}
pd.DataFrame(metrics_data)



,Метрика,Значение
0,RMSE,40.149429
1,MSE,1611.976633
2,R^2 Score,-0.062877
